# EXTRA EXERCISE 2

In a chemical process, data related to the concentration of a given component are measured every two hours. 197 consecutive observations are stored in `concentration.csv` (time series ‘A’ “Time Series Analysis – 3rd edition” Box Jenkins Reinsel – Prentice Hall)  


Estimate the most suitable ARIMA model.

In [ ]:
# Import the necessary libraries
import qdatoolkit as qda
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns

# Import the dataset
data = pd.read_csv('../../Data/concentration.csv')

# Inspect the dataset
data.head()

In [ ]:
# Plot the data 
plt.plot(data['Concentration'], 'o-')
plt.xlabel('Index')
plt.ylabel('Concentration')
plt.title('Time series plot of Concentration')
plt.grid()
plt.show()

In [ ]:
_ = qda.Assumptions(data['Concentration']).independence()


> The process is not stationary. Let's try to use the differencing operation

In [ ]:
data['diff1'] = data['Concentration'].diff(1)

plt.plot(data['diff1'], 'o-')
plt.xlabel('Index')
plt.ylabel('DIFF 1')
plt.title('Time series plot of DIFF 1')
plt.grid()
plt.show()

In [ ]:
_ = qda.Assumptions(data['diff1']).independence()

> After the differencing operation and the ACF/PACF plots, we can try an ARIMA(0, 1, 1) model. Let's try to keep the constant term. 
> 
> <t1 style="color:red"> Remind: parsimony! </t1>

In [ ]:
# fit model ARIMA with constant term
model = qda.ARIMA(data['Concentration'], order=(0,1,1), add_constant=True)

qda.ARIMAsummary(model)

> The constant term is not significant, remove it and fit the model again. 

In [ ]:
# fit model ARIMA without constant term
model = qda.ARIMA(data['Concentration'], order=(0,1,1), add_constant=False)

qda.ARIMAsummary(model)

> The calculated ARIMA model is in the form of an IMA(1,1):
>
> $$Y_t - Y_{t-1} = \nabla Y_t =  \theta_{1}  \epsilon_{t-1} + \epsilon_t $$
>
> Finally, check the assumptions on residuals

In [ ]:
#extract the residuals
residuals = model.resid[1:] 

# Perform the Shapiro-Wilk test
_ = qda.Assumptions(residuals).normality()

In [ ]:
_ = qda.Assumptions(residuals).independence()

In [ ]:
fig, axs = plt.subplots(2, 2)
fig.suptitle('Residual Plots')
stats.probplot(residuals, dist="norm", plot=axs[0,0])
axs[0,0].set_title('Normal probability plot')
axs[0,1].scatter(model.fittedvalues[1:], model.resid[1:])
axs[0,1].set_title('Versus Fits')
fig.subplots_adjust(hspace=0.5)
axs[1,0].hist(residuals)
axs[1,0].set_title('Histogram')
axs[1,1].plot(np.arange(1, len(residuals)+1), residuals, 'o-')
plt.show()

All assumptions are met. Model is adequate. 